<div style="text-align: center;">

  <img src="img/LogoUesc.png" alt="UESC" style="width: 60px; display: block; margin: 0 auto 5px auto;">

  <div style="font-size: 20px;"><strong>UNIVERSIDADE ESTADUAL DE SANTA CRUZ - UESC</strong></div>
  <div style="font-size: 18px;">DEPARTAMENTO DE ENGENHARIAS E COMPUTAÇÃO - DEC</div>
  <div style="font-size: 18px;">Grupo de Pesquisas em Simulações e Controle de Processos</div>

  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

  <div style="font-size: 22px; font-weight: bold; margin-bottom: 10px;">PROGRAMAÇÃO EM PYTHON PARA AS ENGENHARIAS</div>

  <div style="font-size: 18px; margin-top: 10px;"><strong>Tema</strong>: Organização de cálculos de Engenharia utilizando funções</div>
  <div style="font-size: 18px;"><strong>Prof.  Dr.  E. R. Edwards</strong></div>

  <div style="font-size: 16px; margin-top: 15px;">Ilhéus - BA, Abril de 2027</div>
  
  <hr style="margin: 20px 0; border: 1px solid #0074B7;">

</div>

**__Sumário__**

- [1. Por que organizar os cálculos em funções?](#secao-1)
- [2. Decomposição de um problema de Engenharia](#secao-2)
- [3. Funções que chamam outras funções](#secao-3)
- [4. Reutilização de resultados e cálculos repetidos](#secao-4)
- [5. Modelo de Mecânica dos Fluidos](#secao-5)
- [6. Cálculos de Instrumentação](#secao-6)
- [7. Modelos de Controle de Processos](#secao-7)
- [8. Cálculos de Transferência de Calor](#secao-8)
- [9. Funções com listas e laços for](#secao-9)
- [10. Construindo um pequeno programa de Engenharia](#secao-10)
- [11. Exercícios e conclusão](#secao-11)

<div id="secao-1" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  1. <strong>Por que organizar os cálculos em funções?</strong>
</div>

Em problemas reais, uma equação raramente aparece isolada. O cálculo de uma grandeza depende de outras, e um programa extenso, com todas as instruções em sequência, torna-se difícil de verificar. **Organizar o programa em funções significa associar cada etapa matemática a uma tarefa bem definida.**

Neste notebook, vamos partir do fenômeno físico, escrever as equações, resolver um caso numericamente e, somente então, organizar os cálculos em funções. Já conhecemos `def` e `return`; agora estudaremos como as funções colaboram.

<div id="secao-2" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  2. <strong>Decomposição de um problema de Engenharia</strong>
</div>

**Problema:** determinar o regime de escoamento de água em uma tubulação circular. Conhecemos o diâmetro interno $D$, a vazão volumétrica $Q$, a massa específica $\rho$ e a viscosidade dinâmica $\mu$.

As etapas são: calcular a área $A$, determinar a velocidade média $v$ e, por fim, o número de Reynolds $Re$.

$$A=\frac{\pi D^2}{4},\qquad v=\frac{Q}{A},\qquad Re=\frac{\rho vD}{\mu}.$$

Adotando $D=0{,}05\ \mathrm{m}$, $Q=0{,}002\ \mathrm{m^3/s}$, $\rho=1000\ \mathrm{kg/m^3}$ e $\mu=0{,}001\ \mathrm{Pa\,s}$, obtemos $A\approx0{,}0019635\ \mathrm{m^2}$, $v\approx1{,}0186\ \mathrm{m/s}$ e $Re\approx50929{,}58$. O número de Reynolds é adimensional.

In [37]:
import math
D = 0.05
Q = 0.002
rho = 1000
mu = 0.001
A = math.pi * D**2 / 4
v = Q / A
Re = rho * v * D / mu
print(f"Área = {A:.6f} m²")
print(f"Velocidade = {v:.4f} m/s")
print(f"Reynolds = {Re:.2f}")

Área = 0.001963 m²
Velocidade = 1.0186 m/s
Reynolds = 50929.58


O programa anterior funciona, mas mistura três tarefas distintas. Na próxima seção, cada cálculo terá uma função própria.

<div id="secao-3" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  3. <strong>Funções que chamam outras funções</strong>
</div>

Uma função pode chamar outra. Isso permite reproduzir a **dependência entre as equações**: `numero_reynolds()` precisa da velocidade, enquanto `velocidade_media()` precisa da área. Os nomes escolhidos devem indicar o significado físico dos cálculos.

In [38]:
import math

def area_circular(diametro):
    return math.pi * diametro**2 / 4

def velocidade_media(vazao, diametro):
    return vazao / area_circular(diametro)

def numero_reynolds(rho, vazao, diametro, mu):
    velocidade = velocidade_media(vazao, diametro)
    return rho * velocidade * diametro / mu

Re = numero_reynolds(1000, 0.002, 0.05, 0.001)
print(f"Re = {Re:.2f}")

Re = 50929.58


Observe que `numero_reynolds()` não repete a expressão da área. Ela chama `velocidade_media()`, que por sua vez chama `area_circular()`. Para esse exemplo, o valor calculado indica escoamento turbulento segundo os critérios usuais para tubulações circulares.

<div id="secao-4" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  4. <strong>Reutilização de resultados e cálculos repetidos</strong>
</div>

Uma função pode ser reutilizada para diferentes condições de operação. Quando precisamos de **mais de uma grandeza**, também podemos calcular uma vez um resultado intermediário e passá-lo às demais funções, evitando trabalho desnecessário.

In [39]:
def velocidade_por_area(vazao, area):
    return vazao / area

A = area_circular(0.05)
for Q in [0.001, 0.002, 0.003]:
    v = velocidade_por_area(Q, A)
    print(f"Q = {Q:.3f} m³/s -> v = {v:.3f} m/s")

Q = 0.001 m³/s -> v = 0.509 m/s
Q = 0.002 m³/s -> v = 1.019 m/s
Q = 0.003 m³/s -> v = 1.528 m/s


Como o diâmetro não mudou, a área foi calculada uma única vez. Esse cuidado se torna mais importante em simulações com milhares de condições operacionais.

<div id="secao-5" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  5. <strong>Modelo de Mecânica dos Fluidos</strong>
</div>

Vamos acrescentar a **perda de carga distribuída**, representada pela equação de Darcy–Weisbach:

$$\Delta P=f\frac{L}{D}\frac{\rho v^2}{2}.$$

Aqui, $f$ é o fator de atrito de Darcy, $L$ é o comprimento da tubulação e $\Delta P$ é a queda de pressão. Para concentrar o estudo na organização das funções, **adotaremos $f=0{,}03$ como dado**, sem calculá-lo a partir da rugosidade e de Reynolds.

Com $L=10\ \mathrm{m}$ e os dados anteriores, $\Delta P\approx3112{,}6\ \mathrm{Pa}$.

In [40]:
def perda_pressao_darcy(f, comprimento, diametro, rho, velocidade):
    return f * (comprimento / diametro) * rho * velocidade**2 / 2

D, Q, rho, L, f = 0.05, 0.002, 1000, 10, 0.03
v = velocidade_media(Q, D)
delta_p = perda_pressao_darcy(f, L, D, rho, v)
print(f"Queda de pressão = {delta_p:.1f} Pa")

Queda de pressão = 3112.6 Pa


A velocidade é calculada em uma função e utilizada por outra. Note que a equação exige coerência de unidades: metros, segundos, quilogramas e pascais.

<div id="secao-6" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  6. <strong>Cálculos de Instrumentação</strong>
</div>

**Instrumentação:** um transmissor de pressão de 4–20 mA está configurado para medir de 0 a 10 bar. Considerando resposta linear, a conversão é:

$$P=P_{\min}+\frac{I-4}{16}(P_{\max}-P_{\min}).$$

Para $I=12\ \mathrm{mA}$, a pressão é $5\ \mathrm{bar}$. Uma função pode realizar a conversão; outra pode comparar o resultado com o limite operacional.

In [41]:
def corrente_para_pressao(corrente_mA, p_min=0, p_max=10):
    return p_min + (corrente_mA - 4) * (p_max - p_min) / 16

def verificar_pressao(corrente_mA, limite_bar):
    pressao = corrente_para_pressao(corrente_mA)
    return pressao, pressao > limite_bar

pressao, acima_limite = verificar_pressao(12, 6)
print(f"Pressão = {pressao:.2f} bar")
print(f"Acima do limite? {acima_limite}")

Pressão = 5.00 bar
Acima do limite? False


Neste primeiro exemplo, consideramos uma leitura dentro da faixa nominal de 4–20 mA. A validação de sinais fora da faixa será estudada no notebook sobre funções confiáveis.

<div id="secao-7" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  7. <strong>Modelos de Controle de Processos</strong>
</div>

**Controle de Processos:** o erro de controle é definido pela diferença entre a temperatura desejada (*setpoint*) e a temperatura medida no processo:

$$
e = T_{SP} - T_{PV}
$$

Em um controlador proporcional ideal, a contribuição proporcional pode ser calculada por:

$$
u_P = K_c e
$$

Considere uma temperatura desejada de $T_{SP}=80$ °C, uma temperatura medida de $T_{PV}=76$ °C e um ganho proporcional de $K_c=2$ pontos percentuais por °C.

Primeiro, calculamos o erro de controle:

$$
e = 80 - 76 = 4
$$

Portanto, o erro é de **4 °C**. A contribuição proporcional será:

$$
u_P = 2 \times 4 = 8
$$

Assim, a contribuição proporcional corresponde a **8 pontos percentuais**. Esse valor representa apenas a parcela proporcional, e não necessariamente a saída completa de um controlador industrial. Observe que a função `erro_controle()` poderá ser utilizada dentro de `contribuicao_proporcional()`, como mostra a próxima célula de código.

In [42]:
def erro_controle(referencia, medicao):
    return referencia - medicao

def contribuicao_proporcional(kc, referencia, medicao):
    erro = erro_controle(referencia, medicao)
    return kc * erro

print(f"Erro = {erro_controle(80, 76):.1f} °C")
print(f"Contribuição proporcional = {contribuicao_proporcional(2, 80, 76):.1f} %")

Erro = 4.0 °C
Contribuição proporcional = 8.0 %


O cálculo do erro pode ser reutilizado em diferentes estratégias de controle. No futuro, o mesmo raciocínio permitirá organizar modelos de controladores PI e PID.

<div id="secao-8" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  8. <strong>Cálculos de Transferência de Calor</strong>
</div>

**Transferência de Calor:** em regime permanente e condução unidimensional através de uma parede plana, com condutividade constante, a taxa de calor é

$$\dot Q=\frac{kA(T_1-T_2)}{L}.$$

Para $k=0{,}8\ \mathrm{W/(m\,K)}$, $A=2\ \mathrm{m^2}$, $L=0{,}1\ \mathrm{m}$ e $T_1-T_2=30\ \mathrm{K}$, obtemos $\dot Q=480\ \mathrm{W}$. Podemos separar a resistência térmica da determinação da taxa de calor.

In [43]:
def resistencia_termica(k, area, espessura):
    return espessura / (k * area)

def taxa_calor_parede(k, area, espessura, t_quente, t_fria):
    resistencia = resistencia_termica(k, area, espessura)
    return (t_quente - t_fria) / resistencia

q = taxa_calor_parede(0.8, 2.0, 0.1, 60, 30)
print(f"Taxa de calor = {q:.1f} W")

Taxa de calor = 480.0 W


<div id="secao-9" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  9. <strong>Funções com listas e laços for</strong>
</div>

Uma vez definidas, as funções podem ser chamadas repetidamente dentro de um laço `for`. Isso permite avaliar o mesmo modelo sob diferentes condições operacionais, sem copiar suas equações.

In [44]:
temperaturas_frias = [20, 25, 30, 35]
resultados = []
for t_fria in temperaturas_frias:
    q = taxa_calor_parede(0.8, 2.0, 0.1, 60, t_fria)
    resultados.append(q)
    print(f"T_fria = {t_fria:2d} °C -> Q = {q:.1f} W")
print("Resultados (W):", resultados)

T_fria = 20 °C -> Q = 640.0 W
T_fria = 25 °C -> Q = 560.0 W
T_fria = 30 °C -> Q = 480.0 W
T_fria = 35 °C -> Q = 400.0 W
Resultados (W): [640.0, 560.0, 480.0, 400.0]


Observe que a função representa a equação física e o `for` representa a repetição dos cálculos. São responsabilidades diferentes e complementares.

<div id="secao-10" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  10. <strong>Construindo um pequeno programa de Engenharia</strong>
</div>

**Estudo integrado:** vamos organizar um pequeno programa para avaliar uma linha de escoamento em diferentes vazões. O programa deve calcular a velocidade, o número de Reynolds e a queda de pressão. Para cada vazão, a velocidade será calculada apenas uma vez e reutilizada.

In [45]:
def avaliar_tubulacao(q, d, rho, mu, comprimento, fator_atrito):
    area = area_circular(d)
    velocidade = q / area
    reynolds = rho * velocidade * d / mu
    delta_p = perda_pressao_darcy(fator_atrito, comprimento, d, rho, velocidade)
    return velocidade, reynolds, delta_p

for vazao in [0.001, 0.002, 0.003]:
    v, re, dp = avaliar_tubulacao(vazao, 0.05, 1000, 0.001, 10, 0.03)
    print(f"Q={vazao:.3f} m³/s | v={v:.3f} m/s | Re={re:.0f} | ΔP={dp:.0f} Pa")

Q=0.001 m³/s | v=0.509 m/s | Re=25465 | ΔP=778 Pa
Q=0.002 m³/s | v=1.019 m/s | Re=50930 | ΔP=3113 Pa
Q=0.003 m³/s | v=1.528 m/s | Re=76394 | ΔP=7003 Pa


O fator de atrito foi mantido constante por hipótese didática. Em um modelo mais completo, ele dependerá do regime de escoamento e, no caso turbulento, também da rugosidade relativa. Essa evolução será tratada quando estudarmos modelos mais completos e validação de entradas.

<div id="secao-11" style="background-color:#f1faff; border-left:6px solid #0074B7; text-align:left; color:#003366; font-size:22px; font-weight:bold; border-radius:8px; padding:15px 20px; margin-top:20px;">
  11. <strong>Exercícios e conclusão</strong>
</div>

**Exercícios de fixação**

1. Modifique o exemplo de Instrumentação para um transmissor de nível de 0 a 5 m e determine o nível correspondente a 8 mA.
2. Na aplicação de Controle de Processos, avalie a contribuição proporcional para temperaturas medidas de 74, 78, 80 e 83 °C. Explique o significado do sinal.
3. No exemplo de Transferência de Calor, calcule a taxa de calor para espessuras de 0,05, 0,10 e 0,20 m e interprete o resultado.
4. No estudo de tubulações, altere o diâmetro para 0,08 m e compare velocidade, Reynolds e queda de pressão, mantendo as demais hipóteses.

**Conclusão.** Organizar cálculos em funções significa decompor um problema físico em tarefas claras, reutilizar resultados e tornar as hipóteses do modelo mais visíveis. No próximo notebook, estudaremos validação de entradas, documentação e testes para tornar essas funções mais confiáveis.

<div style="text-align: center; font-size: 12px; color: gray;
            margin-top: 40px; padding-top: 15px;
            border-top: 1px solid #dddddd;">

  Material desenvolvido pelo Grupo de Pesquisas em
  Modelagem Computacional da UESC.<br>

  © 2026 — Todos os direitos reservados.
  Uso comercial somente mediante autorização.

</div>